# 15b · GSE232381 · bulk_RNA_seq · scoring these samples on the GSE65391 modules

Reads this study's `expression.rds` and the GSE65391 end product `module_loadings.csv` (for each module:
its genes and their weights; no GSE65391 expression values). Writes `projected_scores.rds` in
`data/run_artifacts/GSE232381/`.

**What is transferred.** The model fitted on the GSE65391 first visits (notebook 05b): for each module,
the genes and their weights (u1 / d1 of the module's first principal component). Scoring a new data set
with a model learned elsewhere is transfer learning by projection (Stein-O'Brien GL et al. Decomposing
cell identity for transfer learning across cellular measurements, platforms, tissues, and species.
*Cell Syst* 2019;8:395-411, doi:10.1016/j.cels.2019.04.004; Sharma G et al. projectR. *Bioinformatics*
2020;36:3592-3593, doi:10.1093/bioinformatics/btaa183).

**Scaling is local.** GSE65391 standardised each gene with its own first-visit mean and SD. These are
array intensities; here the values are RNA sequencing log counts on another scale, so the array's means
and SDs cannot be applied. Each gene is standardised within this study, over its 16 samples, and then
weighted with the array weights:

  score = sum over the module's genes measured here of z times weight

**Coverage.** Genes that this study does not measure are left out, and the share of each module's genes
used is reported.

**Check.** For each module, the projected score against the eigengene recomputed from this study's own
data (notebook 16's "array module" family): Pearson r. A high r means the array's weighting and a local
refit agree on the module's axis in these samples.

## Standardize this study's genes and check coverage

In [1]:
source("../src/paths.R")
x  <- readRDS(art("GSE232381", "expression.rds"))
ld <- read.csv(art("GSE65391", "module_loadings.csv"))
E  <- x$E[, x$meta$sample]
Z  <- (E - rowMeans(E)) / apply(E, 1, sd)                 # genes x samples, standardised in this study
Z  <- Z[apply(E, 1, sd) > 0, ]
cov <- do.call(rbind, lapply(split(ld, ld$module), function(d)
  data.frame(module = d$module[1], genes_in_array_module = nrow(d), genes_here = sum(d$gene %in% rownames(Z)))))
cov$share_used <- round(cov$genes_here / cov$genes_in_array_module, 3)
cov[order(cov$share_used), ]

,module,genes_in_array_module,genes_here,share_used
,<chr>,<int>,<int>,<dbl>
yellow,yellow,383,278,0.726
cyan,cyan,60,48,0.800
green,green,315,271,0.860
brown,brown,647,564,0.872
magenta,magenta,180,158,0.878
pink,pink,212,187,0.882
tan,tan,124,111,0.895
blue,blue,1557,1415,0.909
turquoise,turquoise,3126,2922,0.935


**Explanation**
- `module_loadings.csv` (notebook 05b) holds, for each array module, its genes and their weights.
- `Z`: each gene of this study standardized over its 16 samples. `apply(E, 1, sd) > 0` drops genes that do
  not vary.
- `cov` gives, for each module, how many of its array genes are measured here, and the share.

**Result.** Coverage ranges from 73% (yellow, the ribosomal module: many of its array probes are ribosomal
pseudogenes that the RNA sequencing counts do not include) to 99% (salmon); 13 of 14 modules have 75% or more.

## Score the samples

In [2]:
scores <- sapply(split(ld, ld$module), function(d) {
  d <- d[d$gene %in% rownames(Z), ]
  colSums(Z[d$gene, , drop = FALSE] * d$weight)
})
dim(scores)

[1] 16 14

**Explanation**
- For each module, `colSums(Z[d$gene, ] * d$weight)` sums the standardized genes times their weights: one
  score per sample, as in notebook 05b. Genes not measured here are left out.
- `scores` is samples by modules.

**Check.** Projected score against the locally recomputed eigengene of the same gene list.

## Check against a local refit

In [3]:
suppressMessages(library(WGCNA))
ld_here <- ld[ld$gene %in% rownames(Z), ]
ME_local <- moduleEigengenes(t(E[ld_here$gene, ]), colors = ld_here$module)$eigengenes
check <- data.frame(module = colnames(scores),
                    r_projected_vs_local = sapply(colnames(scores), function(mo) cor(scores[, mo], ME_local[, paste0("ME", mo)])),
                    row.names = NULL)
check$r_projected_vs_local <- round(check$r_projected_vs_local, 3)
check[order(check$r_projected_vs_local), ]

,module,r_projected_vs_local
,<chr>,<dbl>
13,turquoise,0.796
5,green,0.892
1,black,0.965
14,yellow,0.968
12,tan,0.984
8,pink,0.990
3,brown,0.993
10,red,0.993
4,cyan,0.995


**Explanation**
- `moduleEigengenes(t(E[genes, ]), colors = modules)` recomputes each module's eigengene from this study's
  own data, for the same genes.
- `r_projected_vs_local` is the Pearson correlation of the projected score with that local eigengene. A value
  near 1 means the array weights and a local fit agree.

**Result.** For 12 of 14 modules the projected score and the local refit agree closely (r = 0.965 to
0.999), including black (interferon, 0.965), pink (plasma cell, 0.990) and cyan (neutrophil granule, 0.995).
Two agree less: turquoise (0.796) and green (0.892). Turquoise is the module notebook 15 found not
preserved in this study.

## Save

In [4]:
saveRDS(list(scores = scores, coverage = cov, check = check), art("GSE232381", "projected_scores.rds"))

**Explanation**
- `projected_scores.rds` keeps the scores, the coverage and the check.